# Retrain DecisionGator with your corrections

This notebook retrains DecisionGator with the exact recipe of the released model, plus the examples in `data/corrections.jsonl`, and gives you a new model to drop into your bundle folder. See [WRONG-ANSWER.md](https://github.com/CarlFreeAiEngineer/DecisionGator/blob/main/WRONG-ANSWER.md) for the whole story.

**Before you start:** choose an A100 GPU under *Runtime → Change runtime type* (a paid Colab plan). Then run the cells in order. The whole run takes about half an hour.

In [ ]:
# 1. Check the GPU. The recipe needs about 40 GB of GPU memory (an A100).
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the code and data

If you added your corrections to a fork on GitHub, put your fork's address in `REPO`.

In [ ]:
REPO = "https://github.com/CarlFreeAiEngineer/DecisionGator.git"  # or your fork
!git clone --depth 1 {REPO} /content/DecisionGator
%cd /content/DecisionGator
!pip install -q uv

## 3. Add your corrections

Either edit `data/corrections.jsonl` in your fork, or list them here. One example per line in the file; the fields are explained in WRONG-ANSWER.md.

In [ ]:
my_corrections = [
    # {"text": "Do you have an ai product?", "question": "Is this a sales question?", "answer": "yes"},
    # {"text": "...", "question": "Which team should handle this?", "options": ["Billing", "Sales", "HR"], "answer": "HR"},
]

import json
with open("data/corrections.jsonl", "a", encoding="utf-8") as f:
    for item in my_corrections:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
print(open("data/corrections.jsonl", encoding="utf-8").read())
!uv run training/retrain.py --check-only

## 4. Retrain

Downloads the base model, trains for five passes over the data, compresses and calibrates the result, then scores it. About half an hour on an A100.

In [ ]:
!uv run training/retrain.py --name colab

## 5. Results

Held-out test scores next to the released model's, and how the old and new model answer your corrections.

In [ ]:
from IPython.display import Markdown
Markdown(open("runs/colab/results.md", encoding="utf-8").read())

## 6. Download your model

Unzip it into your DecisionGator bundle folder, replacing `model.onnx`, `tokenizer.json` and `manifest.json`.

In [ ]:
from google.colab import files
files.download("runs/colab/decisiongator-model.zip")